# 02 — Train CNN-LSTM on the source dataset, evaluate on source + target

Trains across all seeds in `configs/default.yaml` (`n_seeds`, default 5) for the
statistical-validation protocol in Sec 4.3 of the proposal. Each seed's model is
checkpointed; Level-1 detection metrics (Sec 4.1) are computed on the source test
split and on the full target dataset with **no retraining**.


In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # Anaconda/Windows; harmless elsewhere

import yaml
import numpy as np
import pandas as pd
import torch

from src.utils.seed import set_seed
from src.training.train import get_device, train_model
from src.eval.metrics import compute_detection_metrics, predict
from src.data.dataset import make_dataloaders, FlowDataset
from src.models.cnn_lstm import build_model
from torch.utils.data import DataLoader

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
with open('../data/processed/class_names.yaml') as f:
    class_names = yaml.safe_load(f)

device = get_device(cfg['train']['device'])
print('device:', device)

source = np.load('../data/processed/cicids2017_source.npz')
target = np.load('../data/processed/unsw_nb15_target.npz')
n_features = source['X_train'].shape[1]
n_classes = len(class_names)
os.makedirs('../checkpoints', exist_ok=True)

In [ ]:
results_per_seed = []

for seed in cfg['seeds'][:cfg['n_seeds']]:
    print(f'\n=== seed {seed} ===')
    set_seed(seed)

    splits = {'X_train': source['X_train'], 'y_train': source['y_train'],
              'X_val': source['X_val'], 'y_val': source['y_val'],
              'X_test': source['X_test'], 'y_test': source['y_test']}
    train_loader, val_loader, test_loader = make_dataloaders(
        splits, cfg['data']['sequence_length'], cfg['train']['batch_size'])

    model = build_model(cfg, n_features, n_classes)
    model = train_model(model, train_loader, val_loader, source['y_train'], n_classes, cfg, device)

    torch.save(model.state_dict(), f'../checkpoints/cnn_lstm_seed{seed}.pt')

    # Source test evaluation
    y_true, y_pred, y_proba = predict(model, test_loader, device)
    source_metrics = compute_detection_metrics(y_true, y_pred, y_proba, class_names)
    print('Source test accuracy:', source_metrics['summary']['accuracy'])

    # Target (cross-dataset) evaluation, no retraining
    target_ds = FlowDataset(target['X'], target['y'], cfg['data']['sequence_length'])
    target_loader = DataLoader(target_ds, batch_size=cfg['train']['batch_size'], shuffle=False)
    yt_true, yt_pred, yt_proba = predict(model, target_loader, device)
    target_metrics = compute_detection_metrics(yt_true, yt_pred, yt_proba, class_names)
    print('Target (transfer) accuracy:', target_metrics['summary']['accuracy'])

    results_per_seed.append({
        'seed': seed,
        'source_summary': source_metrics['summary'],
        'target_summary': target_metrics['summary'],
    })

## Aggregate across seeds (mean ± std) — H1 check

In [ ]:
rows = []
for split_name in ['source_summary', 'target_summary']:
    df = pd.DataFrame([r[split_name] for r in results_per_seed])
    agg = df.agg(['mean', 'std']).T
    agg.columns = [f'{split_name}_mean', f'{split_name}_std']
    rows.append(agg)
summary_table = pd.concat(rows, axis=1)
summary_table.to_csv('../results/tables/detection_performance_summary.csv')
summary_table